In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import make_scorer, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1) Daten laden
data = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 2) Target und Features
y = (data["Churn"] == "Yes").astype(int)
X = data.drop(columns=["Churn"])

# 3) Spaltentypen (vereinfachte Heuristik)
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

# 4) Preprocessing (leakage-frei)
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)

# 5) Modelle (Algorithm Selection)
models = {
    "Dummy (MostFrequent)": DummyClassifier(strategy="most_frequent"),
    "LogReg": LogisticRegression(max_iter=200, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "GradBoost": GradientBoostingClassifier(random_state=42)
}

from sklearn.metrics import precision_score, recall_score, f1_score, make_scorer
# 6) Evaluation Setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",  # sinnvoll bei Klassenungleichgewicht
    "f1": make_scorer(f1_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
}
results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "model": name,
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "f1_mean": scores["test_f1"].mean(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean()
    })

pd.DataFrame(results).sort_values(by="f1_mean", ascending=False)

# =========================
# Metriken
# =========================
# Recall: Wie viele echte Kündiger finde ich?
# -> wichtig, wenn ich Kündiger NICHT verpassen will (Frühwarnsystem)
#
# Precision: Wenn ich „kündigt“ sage, wie oft stimmt es?
# -> wichtig, wenn ich wenige falsche Alarme will (nur sichere Fälle ansprechen)
#
# F1-Score: Balance aus Precision und Recall
# -> gute Gesamtmetrik bei Churn (typisch: Klassenungleichgewicht)
#
# ROC-AUC: Wie gut trennt das Modell grundsätzlich Kündiger vs. Nicht-Kündiger?
# -> unabhängig von einem festen Schwellenwert (z.B. 0.5)

# ==================================
# Ergebnis-Tabelle (Interpretation)
# ==================================

# Logistic Regression (LogReg):
# ROC-AUC ~ 0.845 | F1 ~ 0.63 | Recall ~ 0.73 | Precision ~ 0.56
# ✅ Gut: erkennt die meisten Kündiger (höchster Recall) und beste Balance (F1)
# ⚠️ Nachteil: Precision ist moderat -> es gibt einige falsche Alarme

# Gradient Boosting (GradBoost):
# ROC-AUC ~ 0.848 | F1 ~ 0.58 | Precision ~ 0.67 | Recall ~ 0.51
# ✅ Gut: am präzisesten (höchste Precision) und sehr gute Trennschärfe
# ⚠️ Schlecht fürs Frühwarnsystem: übersieht viele Kündiger (niedriger Recall)

# Random Forest:
# ROC-AUC ~ 0.831 | F1 ~ 0.56 | Precision ~ 0.64 | Recall ~ 0.50
# ✅ Solide: erkennt komplexe Muster, Precision ok
# ⚠️ Insgesamt schwächer als LogReg und GradBoost in diesem Setup

# Dummy (MostFrequent):
# ROC-AUC = 0.50 | F1 = 0.00 | Precision = 0.00 | Recall = 0.00
# ❌ Schlecht: findet keine Kündiger (sagt praktisch immer "bleibt")
# ✅ Aber wichtig als Baseline: zeigt, dass echte Modelle deutlich besser sind

,model,roc_auc_mean,f1_mean,precision_mean,recall_mean
1,LogReg,0.844670,0.629789,0.555548,0.727104
3,GradBoost,0.847519,0.577435,0.672372,0.506143
2,RandomForest,0.830787,0.560362,0.635169,0.501842
0,Dummy (MostFrequent),0.500000,0.000000,0.000000,0.000000
